# Football player valuation: a reproducible evaluation
Adetola Adelase · Portfolio revision of the 2021 MSc dissertation

## tl;dr
**Execution status: awaiting the original dataset. No revised model results are claimed.**

This revision separates players across training, validation and test data, compares a median baseline with regression pipelines, and tests whether financial proxies dominate performance. The original dissertation notebook remains unchanged.


## Context & Methods
The target is historical **FIFA/SoFIFA value in EUR millions**, not observed transfer fees. The original notebook is retained at the repository root.

### Key Assumptions
- Input is the original cleaned player table, before feature-dropping cells; monetary fields already use EUR millions.
- All rows for a player stay together, using ID where available, otherwise normalised Name. Name collisions conservatively group distinct people together; inconsistent names can still hide duplicates.
- A deterministic split assigns approximately 60/20/20% of distinct players. Row shares can differ.
- Five-fold grouped CV on training data chooses a model for each feature set by MAE. Validation chooses between feature sets. The final selected workflow is refitted on training + validation and scored once on the untouched test set.
- These are cross-sectional held-out-player results, not future-season forecasting. Historical population coverage and snapshot uncertainty remain limitations.
- Hyperparameters are fixed comparison settings, not an exhaustive tuning exercise. Fold SD is variability across folds, not a confidence interval.


### 1. Setup and source provenance

In [ ]:
from pathlib import Path
import sys, json, platform
import importlib.metadata as metadata

# Supports launching from repository root or notebooks/.
ROOT = Path.cwd().resolve()
if not (ROOT / 'src' / 'valuation.py').exists():
    ROOT = ROOT.parent
assert (ROOT / 'src' / 'valuation.py').exists(), 'Run from repository root or notebooks/'
sys.path.insert(0, str(ROOT / 'src'))
DATA = ROOT / 'data' / 'players_clean.csv'
PROVENANCE = ROOT / 'data' / 'provenance.json'
OUTPUT = ROOT / 'outputs'

# Check real input before importing modelling dependencies.
from valuation import load_data, split_players, select_models, evaluate_final
players, audit = load_data(DATA)
if not PROVENANCE.exists():
    raise FileNotFoundError('Add data/provenance.json; see data/README.md')
provenance = json.loads(PROVENANCE.read_text())
for field in ['source', 'snapshot', 'export_date']:
    if not isinstance(provenance.get(field), str) or not provenance[field].strip():
        raise ValueError(f'Missing provenance field: {field}')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
OUTPUT.mkdir(exist_ok=True)
plt.rcParams.update({'figure.figsize': (9, 5), 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})
GREEN, GREY = '#24664c', '#8b9690'
versions = {p: metadata.version(p) for p in
            ['pandas', 'numpy', 'scikit-learn', 'matplotlib']}
print(json.dumps({'python': platform.python_version(), 'packages': versions,
                  'provenance': provenance, 'input_audit': audit}, indent=2))


## Data
### 2. Validate identity separation and inspect the training population
Only training data informs exploratory charts and model development. Exact duplicate rows and invalid targets/identities are removed with counts recorded in the audit. Missing numeric predictors are imputed inside each fitted pipeline, not before splitting.


In [ ]:
train, validation, test = split_players(players)
split_summary = pd.DataFrame([
    {'split': name, 'rows': len(frame), 'players': frame['_player_group'].nunique()}
    for name, frame in [('train', train), ('validation', validation), ('test', test)]
])
display(split_summary)
display(train[['Value', 'Age', 'Overall', 'Potential', 'Wage', 'Release_Clause']]
        .describe().round(3))
fig, ax = plt.subplots()
ax.hist(train['Value'], bins=40, color=GREEN, edgecolor='white')
ax.set(title='Historical FIFA values in the training split',
       xlabel='FIFA value (EUR millions)', ylabel='Player records')
fig.tight_layout()
plt.show()


## Results
### 3. Compare complete pipelines using training-only cross-validation
Both feature sets use the same split and CV folds. Imputation, scaling and polynomial expansion are fitted within each fold. The financial-proxy feature set adds wage and release clause to the attributes model.


In [ ]:
cv_results, finalists, chosen = select_models(train, validation)
display(cv_results.sort_values(['feature_set', 'cv_mae_eur_m']).round(3))
validation_results = pd.DataFrame([
    {'feature_set': name, 'model': info['model'], **info['validation_metrics']}
    for name, info in finalists.items()
])
display(validation_results.round(3))
print('Validation-selected workflow:', chosen, '/', finalists[chosen]['model'])
fig, ax = plt.subplots(figsize=(10, 6))
chart = cv_results.pivot(index='model', columns='feature_set', values='cv_mae_eur_m')
chart.plot.barh(ax=ax, color=[GREEN, GREY])
ax.set(title='Training-only grouped cross-validation: lower MAE is better',
       xlabel='Mean absolute error (EUR millions)', ylabel='')
ax.legend(title='Feature set', loc='best')
fig.tight_layout()
plt.show()


### 4. Evaluate the selected workflow on untouched players
The final model is chosen before this cell. The median baseline is fitted on the same development population. Do not use test results to tune or reselect models; further changes require a new independent evaluation design.


In [ ]:
test_results, predictions = evaluate_final(train, validation, test, finalists, chosen)
display(test_results.round(3))
print('Negative predictions (not clipped):', int((predictions['predicted_eur_m'] < 0).sum()))
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(predictions['actual_eur_m'], predictions['predicted_eur_m'],
                alpha=.35, s=12, color=GREEN)
low = min(0, predictions['predicted_eur_m'].min())
high = max(predictions['actual_eur_m'].max(), predictions['predicted_eur_m'].max())
axes[0].plot([low, high], [low, high], '--', color=GREY, label='Perfect prediction')
axes[0].set(title='Held-out actual vs predicted FIFA value',
            xlabel='Actual value (EUR m)', ylabel='Predicted value (EUR m)')
axes[0].legend()
axes[1].scatter(predictions['predicted_eur_m'], predictions['residual_eur_m'],
                alpha=.35, s=12, color=GREEN)
axes[1].axhline(0, linestyle='--', color=GREY)
axes[1].set(title='Held-out residuals: actual minus predicted',
            xlabel='Predicted value (EUR m)', ylabel='Residual (EUR m)')
fig.tight_layout()
plt.show()


### 5. Inspect where errors concentrate
Value bands use actual test values solely for descriptive error analysis. Small position groups are shown with counts and excluded from the position chart below when they contain fewer than 20 records.


In [ ]:
by_band = predictions.groupby('value_band', observed=True).agg(
    records=('absolute_error_eur_m', 'size'), mae_eur_m=('absolute_error_eur_m', 'mean'))
display(by_band.round(3))
fig, ax = plt.subplots()
by_band['mae_eur_m'].plot.bar(ax=ax, color=GREEN, rot=0)
ax.set(title='Held-out error by actual FIFA value band',
       xlabel='Actual value band (EUR millions)', ylabel='MAE (EUR millions)')
fig.tight_layout()
plt.show()
if 'Position' in predictions:
    by_position = predictions.groupby('Position').agg(
        records=('absolute_error_eur_m', 'size'),
        mae_eur_m=('absolute_error_eur_m', 'mean'))
    display(by_position.sort_values('records', ascending=False).round(3))
    eligible = by_position.loc[by_position['records'] >= 20].sort_values('mae_eur_m')
    if not eligible.empty:
        fig, ax = plt.subplots(figsize=(9, max(4, len(eligible) * .3)))
        eligible['mae_eur_m'].plot.barh(ax=ax, color=GREEN)
        ax.set(title='Held-out position errors (at least 20 records per group)',
               xlabel='MAE (EUR millions)', ylabel='Position')
        fig.tight_layout()
        plt.show()


## Takeaways
### 6. Record results without inventing a conclusion
This cell generates an initial factual readout only after execution. Before publishing findings, inspect the plots, dataset provenance and influential errors. A lower proxy-model error does not establish practical scouting value or explain causation.


In [ ]:
selected_mae = float(test_results.iloc[0]['mae_eur_m'])
baseline_mae = float(test_results.iloc[1]['mae_eur_m'])
print(f'Selected workflow: {chosen} / {finalists[chosen]["model"]}')
print(f'Held-out MAE: EUR {selected_mae:.3f}m; median baseline: EUR {baseline_mae:.3f}m.')
print(f'Absolute MAE reduction versus baseline: EUR {baseline_mae - selected_mae:.3f}m.')
print('Interpret alongside value-band errors, snapshot limits and financial-proxy risk.')
cv_results.to_csv(OUTPUT / 'training_cv.csv', index=False)
validation_results.to_csv(OUTPUT / 'validation_comparison.csv', index=False)
test_results.to_csv(OUTPUT / 'test_metrics.csv', index=False)
by_band.to_csv(OUTPUT / 'test_error_by_value_band.csv')
# Do not export individual player records by default.
(OUTPUT / 'run_manifest.json').write_text(json.dumps({
    'source': provenance, 'audit': audit, 'seed': 42, 'versions': versions,
    'split': split_summary.to_dict('records'), 'selected_features': chosen,
    'selected_model': finalists[chosen]['model'],
}, indent=2))


### Limitations and follow-up
- Historical game valuations cannot validate real transfer-fee recommendations.
- Missing original snapshot/version limits replication; record what is known without guessing.
- A random player split does not test future seasons or unseen leagues.
- Financial proxies may encode the same valuation system as the target.
- Repeat identities with inconsistent names may survive name-based grouping; prefer stable player IDs.
- Raw euro-value regression can produce negative predictions. They are counted and left visible, rather than silently clipped.
- This revision is unexecuted until the original data and environment are restored. No output tables or figures are fabricated.

Sources: original notebook blob `eddc195d71c8730116e7c6091c1477a32d9e0393`; scikit-learn documentation for Pipeline, GroupKFold, DummyRegressor and regression metrics.
